In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import pickle
import matplotlib.pyplot as plt
from scipy.stats import lognorm, norm, describe, skew, kurtosis

from hazrisk.risk import numerical_mafe
from hazrisk.hazard import mafe_to_poe

from standes.analysis.recorders import get_recorder, get_recorders
from standes.fragility_curves import fragility_from_msa, fragility_from_ida, FragilityCurve
from standes.intensitymeasures import AverageSpectralAcceleration
from phd_project.config import config

cfg = config.load_config()

In [38]:
N_STOREYS = [3]
ANALYSIS_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]
RESULTS_ROOT = cfg["results"]["site_fragility_curves"]
FRAG_ROOT = cfg["proc_data"]["wp1_sites_fragility_curves"]
STRIPE_IML_PATH = Path(r"C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\imls_for_selection_AvgSA_03.json")
HAZ_CURVE_PATH = cfg["proc_data"]["AvgSA_03_hazard_curves_4sig"]
OUT_FOLDER = cfg["proc_data"]["wp1_sites_fragility_curves"] / "msa_ida_fema_comparison"

G = 9810    # mm/s²

PC_UPPER_THRESHOLD = 0.7 # the minimum upper collapse probabilty to consider a fragility curve as well defined.
PC_LOWER_THRESHOLD = 0.2 # the maximum lower collapse probabilty to consider a fragility curve as well defined.

# im
IM = AverageSpectralAcceleration(0, 3, n_periods=10)

# generate the N-bootstrap samples
K_SAMPLES = 1000
N_STRIPE_RECS = 30          # number of records per stripe
N_IDA_RECS = 22      # number of records in the ida
ALPHA = 0.05    # significance level for hypothesis test
EC8_TARGET = 2e-4

def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"

In [39]:
# Load hazard curves
with open(HAZ_CURVE_PATH, "rb") as file:
    hcs = pickle.load(file)

# Load the stripe IMLs for the MSA
with open(STRIPE_IML_PATH, "r") as file:
    stripe_imls = json.load(file)

## Load Fragility Curves

In [40]:
sites = list(range(0, 60))
fragility_curves = {}

for site in sites:#[34:35]:
    site_fcs = {}
    for ns in N_STOREYS:
        structure_fcs = {}
        # building tag
        tag = structure_tag(site, ns)

        # load MSA fragility - load from processed data
        fc_path =  FRAG_ROOT / f"site_{site}" / f"{tag}_msa_collapsefragility_AvgSA_03.json"
        try:    
            with open(fc_path, "r") as file:
                fc_msa = json.load(file)
                fc_msa = {k: np.array(v) if isinstance(v, list) else v for k, v in fc_msa.items()}
        except FileNotFoundError:
            print(f"No Fragility Curve for MSA: site {site} and {ns}s. Skipping...")
            continue

        structure_fcs["msa"] = fc_msa

        # load FEMA-P695 IDA fragility - load from processed data
        fc_path =  FRAG_ROOT / f"site_{site}" / f"{tag}_ida_femap695_collapsefragility_AvgSA_03.json"
        try:    
            with open(fc_path, "r") as file:
                fc_p695 = json.load(file)
                fc_p695 = {k: np.array(v) if isinstance(v, list) else v for k, v in fc_p695.items()}
        except FileNotFoundError:
            print(f"No Fragility Curve for IDA-FEMAP695: site {site} and {ns}s. Skipping...")
            continue
        
        structure_fcs["ida-femap695"] = fc_p695
        site_fcs[tag] = structure_fcs

    fragility_curves[site] = site_fcs

No Fragility Curve for MSA: site 24 and 3s. Skipping...
No Fragility Curve for MSA: site 31 and 3s. Skipping...
No Fragility Curve for MSA: site 38 and 3s. Skipping...


## Uncertainty Estimation

There is uncertainty associated with the analysis method and the subsequent fitting of the fragility curves. To be able to decide of the differences between the IDA- and MSA-derived fragility curves are significant we need to compare the distributions of the median and dispersion values. Using parametric resampling we can quantify the distribution of these quantities. The parameteric resampling method used in described by Iervolino (2022): "Estimation uncertainty for some common seismic fragility curve fitting methods."

In [41]:
def bootstrap_msa(fc_msa: dict, n_tests: int, k_samples: int, im=IM, seed=1) -> list[FragilityCurve]:
    ### MSA simulations
    rng = np.random.default_rng(seed)

    collapses = []
    for iml in fc_msa["efc"][0, :]:
        p = lognorm.cdf(iml, s=fc_msa["dispersion"], scale=fc_msa["median"])

        im_collapse_samples = rng.binomial(n_tests, p, size=k_samples)
        collapses.append(im_collapse_samples)

    ims = fc_msa["efc"][0, :]
    collapses = np.column_stack(collapses)

    sim_fcs_msa = []
    for row in collapses:
        sim_fcs_msa.append(fragility_from_msa(ims, row, n_tests, im))

    return sim_fcs_msa


def bootstrap_ida(fc_ida, n_recs, k_samples, im, seed=1):
    ### IDA FEMA-P695 simulations
    rng = np.random.default_rng(seed)
    # sample the N_ida points randomly from this distribution (22-points) for k_trials
    collapse_imls = rng.lognormal(np.log(fc_ida["median"]), fc_ida["dispersion"], size=(k_samples, n_recs))

    # fit the distribution of the median and dispersion
    sim_fcs_ida = []
    for row in collapse_imls:
        sim_fcs_ida.append(fragility_from_ida(row, im))

    return sim_fcs_ida

def plot_fragility_curves(ax, fc_msa, fc_ida, sim_fcs_msa, sim_fcs_ida):
    im_max = max([lognorm.ppf(0.95, s=fc.dispersion, scale=fc.median) for fc in sim_fcs_msa])
    imls = np.linspace(0, im_max, 75) # in g

    fcs = []
    for sfc in sim_fcs_ida:
        fcs.append(lognorm.cdf(imls, s=sfc.dispersion, scale=sfc.median))
    fcs = np.column_stack(fcs)
    ida_bounds = np.column_stack([np.min(fcs, axis=1), np.max(fcs, axis=1)]) 

    fcs = []
    for sfc in sim_fcs_msa:
        fcs.append(lognorm.cdf(imls, s=sfc.dispersion, scale=sfc.median))
    fcs = np.column_stack(fcs)
    msa_bounds = np.column_stack([np.min(fcs, axis=1), np.max(fcs, axis=1)]) 

    base_ida_fc = lognorm.cdf(imls, s=fc_ida["dispersion"], scale=fc_ida["median"])
    base_msa_fc = lognorm.cdf(imls, s=fc_msa["dispersion"], scale=fc_msa["median"])
    ax.fill_between(imls, ida_bounds[:, 0], ida_bounds[:, 1], color="r", alpha=0.2, label="IDA variation")
    ax.fill_between(imls, msa_bounds[:, 0], msa_bounds[:, 1], color="b", alpha=0.2, label="MSA variation")
    ax.plot(imls, base_ida_fc, color="r", ls="-", lw=2, label="IDA-FEMA-P695")
    ax.plot(imls, base_msa_fc, color="b", ls="-", lw=2, label="MSA")
    ax.grid(ls="-.", color="0.8")
    ax.set_xlim(0)

    ax.set_ylabel("Probability of Collapse, P[C]")
    ax.set_xlabel("AvgSA[0,3], in [g]")

    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")

    return imls


def get_stats(values) -> dict[str, float]:
    return {"N_obs": len(values),
            "min": min(values),
            "max": max(values),
            "mean": np.mean(values),
            "median": np.median(values),
            "variance": np.var(values),
            "stderror": np.std(values),
            "2.5pc": np.percentile(values, 2.5),
            "5pc": np.percentile(values, 5),
            "16pc": np.percentile(values, 16),
            "84pc": np.percentile(values, 84),
            "95pc": np.percentile(values, 95),
            "97.5pc": np.percentile(values, 97.5),
            "skewness": skew(values),
            "kurtosis": kurtosis(values)
            }

def _print_stats(stat_dicts: list[dict[str, float]], headings: list[str]|None=None):

    title_string = f"{"Stat":12}"
    if headings:
        for heading in headings:
            title_string += f"{heading:>12}"
    title_string += "\n" + "-" * (len(stat_dicts) + 1) * 12
    
    keys = stat_dicts[0].keys()
    values = [d.values() for d in stat_dicts]

    value_string = ""
    for z in zip(keys, *values):
        value_string += f"{z[0]:12}"
        if any(np.array(z[1:]) < 0.001):
            for vi in z[1:]:
                value_string += f"{vi:12.3e}"
        else:
            for vi in z[1:]:
                value_string += f"{vi:12.3f}"
        value_string += "\n"

    print(title_string)
    print(value_string)


def plot_theta_beta_hists(ax1, ax2, thetas_msa, betas_msa, thetas_ida, betas_ida, n_bins=20):
    ax1.hist(thetas_msa, label="MSA", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax1.hist(thetas_ida, label="IDA", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax2.hist(betas_msa, label="MSA", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax2.hist(betas_ida, label="IDA", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax1.set_xlabel(r"Median Collapse Capacity, $\theta$")
    ax2.set_xlabel(r"Dispersion, $\beta$")
    ax1.set_ylabel("PDF")

    leg = ax1.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")

    leg = ax2.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def squared_mahalanobis_distance(d):
    d_bar = np.mean(d, axis=0)    # the mean point
    Gamma = 1 / (len(d)-1) * (d - d_bar).T@(d - d_bar)    # the covariance matrix
    Gamma_inv = np.linalg.inv(Gamma)

    # calculate the Mahalanobis distance for each element
    D2Mis = np.sum((d - d_bar) @  Gamma_inv * (d - d_bar), axis=1)
    # calculate the null
    D2M_test_point = (d_bar.reshape((1,2)) @ Gamma_inv @ d_bar.reshape((2,1)))[0][0]

    return D2Mis, d_bar, Gamma, D2M_test_point


def hypothesis_test(dis, d_orig, alpha, k_samples):
    # calculate difference vectors (sample statistics)
    D2Mis, d_bar, Gamma, D2M_test_point = squared_mahalanobis_distance(dis)
    D2M_orig = (d_orig.reshape((1,2)) @ np.linalg.inv(Gamma) @ d_orig.reshape((2,1)))[0][0]

    # - bias correction for BC confidence interval
    q = np.mean(D2Mis < D2M_orig)   # proportion less than the mean value
    z0 = norm.ppf(q)    # bias correction

    CI_bc = norm.cdf(2 * z0 + norm.ppf(1-alpha))

    D2Mis_sorted = sorted(D2Mis)
    idx = int(CI_bc * k_samples)
    CI_distance = D2Mis_sorted[idx]

    # achieved significance level
    asl = np.mean(D2Mis >= D2M_test_point)

    return D2Mis, d_bar, Gamma, D2M_test_point, q, z0, CI_bc, CI_distance, asl


def get_CI_ellipse(d_bar, CI_distance, Gamma):
    phi = np.linspace(0, 2 * np.pi, 200)
    unit_circle = np.vstack((np.cos(phi), np.sin(phi))) # Shape: (2, 200)

    L = np.linalg.cholesky(Gamma) # compute Cholesky factor L
    CI_ellipse = d_bar[:, None] + np.sqrt(CI_distance) * (L @ unit_circle) #scale/rotate unit circle
    return CI_ellipse


def visualise_hypothesis_test(ax, D2Mis, alpha, D2M_test_point, CI_distance, n_bins=30):
    ax.hist(D2Mis, density=True, facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], bins=n_bins)
    ax.axvline(CI_distance, color='k', linewidth=1.5, ls="--", label=rf"$\alpha = {alpha:.2f}$ CI (BC) Limit")
    ax.axvline(D2M_test_point, color='r', linewidth=1.5, ls="--", label=f"Test Distance")
    ax.axvspan(0, CI_distance, color="g", alpha=0.3, label="$H_0$ cannot be rejected")

    ax.set_ylabel("PDF")
    ax.set_xlabel("$D^2_M$, [-]")

    ax.set_xlim(0)
    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def plot_msa_ida_diff(ax, dis, d_bar, d_orig, CI_distance, Gamma):

    CI_ellipse = get_CI_ellipse(d_bar, CI_distance, Gamma)
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.plot(dis[:, 0], dis[:, 1], marker=".", mfc="g", mec="g", ls="none", label="Bootstr. vals")
    ax.plot(d_bar[0], d_bar[1], marker="o", mfc="r", mec="k", ls="none", label="Mean")
    ax.plot(d_orig[0], d_orig[1], marker="o", mfc="b", mec="k", ls="none", label="Orig. estimate")
    ax.plot(CI_ellipse[0, :], CI_ellipse[1, :], ls="-", color="k", label="Sig. limit")
    ax.set_ylabel(r"$\Delta_{\ln{\beta}} = \ln{\beta_{MSA}} - \ln{\beta_{IDA}}$")
    ax.set_xlabel(r"$\Delta_{\ln{\theta}} = \ln{\theta_{MSA}} - \ln{\theta_{IDA}}$")
    
    ax.grid(ls="-.", color="0.8")
    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def plot_theta_beta_correlation(ax, thetas_msa, betas_msa, thetas_ida, betas_ida):
    ax.plot(thetas_msa, betas_msa, marker=".", mfc="b", mec="b", ls="none", label="MSA")
    ax.plot(thetas_ida, betas_ida, marker=".", mfc="r", mec="r", ls="none", label="IDA-FEMA-P695")

    ax.set_ylabel(r"$\beta$")
    ax.set_xlabel(r"$\theta$")
    ax.grid(ls="-.", color="0.8")
    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def plot_msa_ida_ratios(ax1, ax2, theta_ratio, beta_ratio, n_bins):
    ax1.hist(theta_ratio, label=r"$\theta_{MSA}/\theta_{IDA}$", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax2.hist(beta_ratio, label=r"$\beta_{MSA}/\beta_{IDA}$", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax1.set_xlabel(r"$\theta_{MSA}/\theta_{IDA}$")
    ax2.set_xlabel(r"$\beta_{MSA}/\beta_{IDA}$")
    ax1.set_ylabel("PDF")
    ax2.set_ylabel("PDF")

    leg = ax1.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")
    leg = ax2.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def calculate_mafcs(sim_fcs: list[FragilityCurve], haz_curve, imls):
    mafcs = []
    for sim_fc in sim_fcs:
        fc = lognorm.cdf(imls, s=sim_fc.dispersion, scale=sim_fc.median)
        mafcs.append(numerical_mafe(haz_curve, fc))
    return np.array(mafcs)


def plot_mafc_distributions(ax, mafcs_msa, mafcs_ida, n_bins=20):
    ax.hist(mafcs_msa, label="MSA", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax.hist(mafcs_ida, label="IDA", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax.set_xlabel(r"$\lambda_C$")
    ax.set_ylabel("PDF")

    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def calculate_mafc_margins(mafcs_msa, mafcs_ida, target):
    # % differences
    MAFC_diff = (mafcs_msa - mafcs_ida)
    # relative to MSA
    mafc_rel_msa = MAFC_diff / mafcs_msa

    # relative to EC8 target
    mafc_margin_msa = (target / mafcs_msa) #/ target
    mafc_margin_ida = (target / mafcs_ida)
    return mafc_margin_msa, mafc_margin_ida, mafc_rel_msa


def plot_mafc_difference_distributions(ax1, ax2, mafc_rel_MSA, mafc_margin_msa, mafc_margin_ida, n_bins):
    ax1.hist(mafc_rel_MSA, label=r"$\Delta\lambda_{C,%MSA}$", facecolor=[(1.0, 0.0, 1.0, 0.5)], edgecolor=[(1.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax2.hist(mafc_margin_msa, label=r"MSA", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax2.hist(mafc_margin_ida, label=r"IDA", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax1.set_xlabel(r"Difference relative to MSA, $\Delta\lambda_{C,MSA}$")
    ax2.set_xlabel(r"EC8 Margin, $\lambda_{C,EC8} / \lambda_{C,x}$")
    ax1.set_ylabel("PDF")
    ax2.set_ylabel("PDF")

    leg = ax2.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")


def plot_pc_distributions(ax, pcs_msa, pcs_ida, t, n_bins=20):
    ax.hist(pcs_msa, label="MSA", facecolor=[(0.0, 0.0, 1.0, 0.5)], edgecolor=[(0.0, 0.0, 1.0, 1.0)], density=True, bins=n_bins)
    ax.hist(pcs_ida, label="IDA", facecolor=[(1.0, 0.0, 0.0, 0.5)], edgecolor=[(1.0, 0.0, 0.0, 1.0)], density=True, bins=n_bins)

    ax.set_xlabel(rf"P[collapse] in {t} years")
    ax.set_ylabel("PDF")

    leg = ax.legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")

In [42]:
def process_structure(out_folder, site, nstoreys, fc_msa, n_recs_msa, fc_ida, n_recs_ida, k_samples, im, haz_curve,
                      alpha=0.05, ec8_target = 2e-4, n_bins=20, t=50, seed=1):
    # comparison between MSA and IDA
    # fragility curves + boot straps
    sim_fcs_msa = bootstrap_msa(fc_msa, n_recs_msa, k_samples, im, seed=seed)
    sim_fcs_ida = bootstrap_ida(fc_ida, n_recs_ida, k_samples, im, seed=seed)

    ### MSA
    thetas_msa = [fc.median for fc in sim_fcs_msa] 
    betas_msa = [fc.dispersion for fc in sim_fcs_msa] 
    v_msa = np.column_stack([np.log(thetas_msa), np.log(betas_msa)])

    ### IDA
    thetas_ida = [fc.median for fc in sim_fcs_ida] 
    betas_ida = [fc.dispersion for fc in sim_fcs_ida] 
    v_ida = np.column_stack([np.log(thetas_ida), np.log(betas_ida)])

    ### Statistical significance
    dis = v_msa - v_ida     # vector of differences
    d_orig = np.log(np.array([fc_msa["median"], fc_msa["dispersion"]])) - np.log(np.array([fc_ida["median"], fc_ida["dispersion"]]))
    D2Mis, d_bar, Gamma, test_distance, q, z0, CI_bc, CI_distance, asl = hypothesis_test(dis, d_orig, alpha, k_samples)

    ### Plotting
    fig1, axs1 = plt.subplots(3, 2, figsize=(10, 12))
    plt.close(fig1)
    
    imls = plot_fragility_curves(axs1[0, 0], fc_msa, fc_ida, sim_fcs_msa, sim_fcs_ida)
    plot_theta_beta_correlation(axs1[0, 1], thetas_msa, betas_msa, thetas_ida, betas_ida)
    plot_theta_beta_hists(axs1[1, 0], axs1[1, 1], thetas_msa, betas_msa, thetas_ida, betas_ida)
    visualise_hypothesis_test(axs1[2, 0], D2Mis, alpha, test_distance, CI_distance)
    plot_msa_ida_diff(axs1[2, 1], dis, d_bar, d_orig, CI_distance, Gamma)

    fig1.suptitle(f"Site {site} - {nstoreys} stories - MSA vs. IDA (Chart 1/2)",
                  fontweight="bold")
    fig1.tight_layout()

    ### Other stats
    theta_ratio = np.array(thetas_msa) / np.array(thetas_ida)
    beta_ratio = np.array(betas_msa) / np.array(betas_ida)

    combined_imls = sorted(np.concat([haz_curve[:,0].flatten(), imls]))
    hc_interp = np.interp(combined_imls, hc[:, 0], hc[:, 1])
    mafcs_msa = calculate_mafcs(sim_fcs_msa, hc_interp, combined_imls)
    mafcs_ida = calculate_mafcs(sim_fcs_ida, hc_interp, combined_imls)
    pcs_msa = mafe_to_poe(mafcs_msa, t) * 100        # in %
    pcs_ida = mafe_to_poe(mafcs_ida, t) * 100        # in %
    mafc_margin_msa, mafc_margin_ida, mafc_rel_msa = calculate_mafc_margins(mafcs_msa, mafcs_ida, ec8_target)

    fig2, axs2 = plt.subplots(3, 2, figsize=(10, 12))
    plt.close(fig2)
    plot_msa_ida_ratios(axs2[0, 0], axs2[0, 1], theta_ratio, beta_ratio, n_bins)
    plot_mafc_distributions(axs2[1, 0], mafcs_msa, mafcs_ida, n_bins)
    plot_pc_distributions(axs2[1, 1], pcs_msa, pcs_ida, t, n_bins)
    plot_mafc_difference_distributions(axs2[2, 0], axs2[2, 1], mafc_rel_msa, mafc_margin_msa, mafc_margin_ida, n_bins)

    fig2.suptitle(f"Site {site} - {nstoreys} stories - MSA vs. IDA (Chart 2/2)", 
                  fontweight="bold")
    fig2.tight_layout()
    

    ## create stats dicts
    stats_dicts = {
        "theta_msa": get_stats(thetas_msa),
        "beta_msa": get_stats(betas_msa),
        "theta_ida": get_stats(thetas_ida),
        "beta_ida": get_stats(betas_ida),
        "D2Mis": get_stats(D2Mis),
        "theta_ratio": get_stats(theta_ratio),
        "beta_ratio": get_stats(beta_ratio),
        "mafc_msa": get_stats(mafcs_msa),
        "mafc_ida": get_stats(mafcs_ida),
        "pcs_msa": get_stats(pcs_msa),
        "pcs_ida": get_stats(pcs_ida),
        "mafc_margin_msa": get_stats(mafc_margin_msa),
        "mafc_margin_ida": get_stats(mafc_margin_ida),
        "mafc_rel_msa": get_stats(mafc_rel_msa),
    }

    bootstrap_stats = pd.DataFrame.from_dict(stats_dicts, orient="columns")

    ## dict of remaining stats
    summary_stats = {
        "statistically_different": test_distance >= CI_distance,
        "rho_msa": np.corrcoef(thetas_msa, betas_msa)[0,1],
        "rho_ida": np.corrcoef(thetas_ida, betas_ida)[0,1],
        "msa_pt_estimate": [fc_msa["median"], fc_msa["dispersion"]],
        "ida_pt_estimate": [fc_ida["median"], fc_ida["dispersion"]],
        "d_orig": d_orig,
        "d_bar": d_bar, 
        "covar": Gamma, 
        "test_distance": test_distance, 
        "q": q, 
        "z0_bias_correction": z0, 
        "bias_corr_CI": CI_bc, 
        "CI_distance": CI_distance, 
        "achieved_sig_level": asl,
    }
    
    ## save figures
    tag = structure_tag(site, nstoreys)
    fig1.savefig(out_folder / f"{tag}_chart1_hypothesis_test.jpg")
    fig2.savefig(out_folder / f"{tag}_chart2_risk_metrics.jpg")

    return bootstrap_stats, summary_stats


In [51]:
bootstrap_stats = {}
summary_stats = {}

site = 23
site2 = 24
ii = 4


for site in list(fragility_curves.keys())[site:site2+1]:
    hc = hcs[site]["AvgSA"]["mean"]
    for ns in N_STOREYS:
        tag = structure_tag(site, ns)
        # get the fragility curve estimates
        try:
            fc_msa = fragility_curves[site][tag]["msa"]
            fc_ida = fragility_curves[site][tag]["ida-femap695"]
        except KeyError:
            continue

        # process the structure
        bootstrap_stats[tag], summary_stats[tag] = process_structure(OUT_FOLDER, site, ns, fc_msa, N_STRIPE_RECS, fc_ida, N_IDA_RECS, K_SAMPLES, IM, hc, ALPHA, EC8_TARGET) 

summary_stats = pd.DataFrame.from_dict(summary_stats, orient="index")
bootstrap_stats = pd.concat(bootstrap_stats).unstack()

with open(OUT_FOLDER / "summary_stats.pickle", "wb") as file:
    pickle.dump(summary_stats, file)

bootstrap_stats.to_csv(OUT_FOLDER / "bootstrap_stats.csv")

C:\Users\clemettn\AppData\Local\Temp\ipykernel_74896\626221625.py:11: RuntimeWarning: divide by zero encountered in log
  v_msa = np.column_stack([np.log(thetas_msa), np.log(betas_msa)])
C:\Users\clemettn\AppData\Local\Temp\ipykernel_74896\2364727173.py:136: RuntimeWarning: invalid value encountered in subtract
  Gamma = 1 / (len(d)-1) * (d - d_bar).T@(d - d_bar)    # the covariance matrix
C:\Users\clemettn\AppData\Local\Temp\ipykernel_74896\2364727173.py:136: RuntimeWarning: invalid value encountered in matmul
  Gamma = 1 / (len(d)-1) * (d - d_bar).T@(d - d_bar)    # the covariance matrix
C:\Users\clemettn\AppData\Local\Temp\ipykernel_74896\2364727173.py:140: RuntimeWarning: invalid value encountered in subtract
  D2Mis = np.sum((d - d_bar) @  Gamma_inv * (d - d_bar), axis=1)
c:\Users\clemettn\Documents\phd\.venv\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2305: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
c:\Users\clemettn\Document

ValueError: autodetected range of [nan, nan] is not finite

In [44]:
site

23